# ML Coding — Day 7: MoE II (backward & load balancing)

**~1 hour, vectorized NumPy.** The training side of mixture-of-experts: the load-balancing loss and its
gradient, the router/combine backward, and capacity-based dispatch. Gradients are checked against
finite differences (`_numgrad`). No element loops in solutions. Run the **helpers cell first**.

**Q1** top-1 routing `[warmup]` · **Q2** Switch aux loss `[medium]` · **Q3** aux-loss gradient
`[medium]` · **Q4** MoE combine fwd+bwd `[hard]` · **Q5** capacity dispatch via cumsum `[hard · trick]`.

---

In [ ]:
# --- helpers (run me first) ---
import numpy as np


def _softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def _numgrad(f, x, eps=1e-6):
    x = np.asarray(x, float)
    grad = np.zeros_like(x)
    flat, gflat = x.reshape(-1), grad.reshape(-1)
    for i in range(flat.size):
        o = flat[i]
        flat[i] = o + eps; a = f(x)
        flat[i] = o - eps; b = f(x)
        flat[i] = o
        gflat[i] = (a - b) / (2 * eps)
    return grad

## Q1 — Top-1 routing  ·  `[warmup]`

**Papers:** Shazeer 2017 / Fedus 2021 (Switch). `route_top1(gate_logits, num_experts) ->
(assignments, counts)`: each token goes to its argmax expert; return the per-token assignment and the
per-expert token counts.

**Lock down:** `argmax` over the expert axis; `np.bincount(..., minlength=num_experts)` for counts.

In [ ]:
import numpy as np


def route_top1(gate_logits, num_experts):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    g = np.array([[1., 0, 0], [0, 2., 0], [0, 0, 3.], [5., 0, 0]])
    a, c = route_top1(g, 3)
    assert list(a) == [0, 1, 2, 0] and list(c) == [2, 1, 1]
    print("Q1 OK")

_q1()

## Q2 — Switch load-balancing auxiliary loss  ·  `[medium]`

**Paper:** Fedus et al., *Switch Transformer*, 2021 (arXiv:2101.03961). **Why it matters:** without it,
the router collapses to a few experts. The aux loss pushes routing toward uniform:
`L = E · Σ_e f_e · P_e`, where `f_e` = fraction of tokens (top-1) routed to expert `e` and `P_e` = mean
softmax gate probability for `e`. It's minimized (= 1) at perfectly balanced routing.

`aux_loss(gate_logits, num_experts) -> float`. No loops.

In [ ]:
def aux_loss(gate_logits, num_experts):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(0)
    assert aux_loss(rng.standard_normal((100, 4)), 4) > 0
    assert np.isclose(aux_loss(np.zeros((100, 4)), 4), 1.0)   # uniform probs, all argmax->0: 4*(1*0.25)
    print("Q2 OK")

_q2()

## Q3 — Aux-loss gradient  ·  `[medium]`

`aux_loss_backward(gate_logits, num_experts) -> grad`. The hard assignment `f_e` is treated as a
**constant** (argmax has zero gradient a.e.), so the gradient flows only through `P_e`:
`∂L/∂logit[n,c] = (E/N) · p[n,c] · (f_c − Σ_e f_e·p[n,e])`.

**Why finite-diff works here:** for generic logits the argmax is locally fixed, so `f_e` really is
constant under tiny perturbations — the numeric gradient of the full loss matches. No loops.

In [ ]:
def aux_loss_backward(gate_logits, num_experts):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(1)
    g = rng.standard_normal((20, 4))
    grad = aux_loss_backward(g, 4)
    assert np.allclose(grad, _numgrad(lambda x: aux_loss(x, 4), g), atol=1e-5)
    print("Q3 OK")

_q3()

## Q4 — MoE combine: forward + backward  ·  `[hard]`

The soft-combine at the heart of a (dense) MoE layer: `y[n] = Σ_e p[n,e]·expert_out[e,n]`, with
`p = softmax(gate_logits)`. `gate_logits` is `(N,E)`, `expert_outs` is `(E,N,D)`.

Implement `moe_combine(gate_logits, expert_outs) -> y (N,D)` and
`moe_combine_backward(dy, gate_logits, expert_outs) -> (dgate_logits, dexpert_outs)`.

**Hint:** `dexpert = einsum('ne,nd->end', p, dy)`; `dp = einsum('end,nd->ne', expert_outs, dy)`; then
`dgate = softmax_backward(dp, p)` over the expert axis. No loops.

In [ ]:
def moe_combine(gate_logits, expert_outs):
    raise NotImplementedError


def moe_combine_backward(dy, gate_logits, expert_outs):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(2)
    N, E, D = 4, 3, 5
    g = rng.standard_normal((N, E)); eo = rng.standard_normal((E, N, D))
    y = moe_combine(g, eo)
    assert y.shape == (N, D)
    dy = rng.standard_normal((N, D))
    dgate, deo = moe_combine_backward(dy, g, eo)
    assert np.allclose(dgate, _numgrad(lambda x: np.sum(dy * moe_combine(x, eo)), g), atol=1e-5)
    assert np.allclose(deo, _numgrad(lambda x: np.sum(dy * moe_combine(g, x)), eo), atol=1e-5)
    print("Q4 OK")

_q4()

## Q5 — Capacity dispatch via cumsum  ·  `[hard · tensor trick]`

**Paper:** GShard (Lepikhin et al. 2020, arXiv:2006.16668). Each expert has a fixed **capacity**; tokens
beyond it (in arrival order) are **dropped**. `dispatch(gate_logits, capacity) -> (keep, expert, slot)`:
`expert` = top-1 expert per token; `slot` = that token's 0-based position among tokens routed to the
same expert; `keep = slot < capacity`.

**The trick:** compute every token's slot with **one cumsum** over the routing one-hot — `slot =
(cumulative count of its expert up to and including it) − 1`. No Python loops.

In [ ]:
def dispatch(gate_logits, capacity):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    g = np.array([[2., 0], [2., 0], [0, 2.], [2., 0], [0, 2.]])   # experts: 0,0,1,0,1
    keep, expert, slot = dispatch(g, capacity=2)
    assert list(expert) == [0, 0, 1, 0, 1]
    assert list(slot) == [0, 1, 0, 2, 1]                          # expert 0: slots 0,1,2
    assert list(keep) == [True, True, True, False, True]          # 3rd token to expert 0 dropped
    print("Q5 OK")

_q5()

## Likely follow-ups
- Top-2 routing (Switch uses 1, GShard 2); the second-expert gate and its aux term.
- Router z-loss for stability; expert dropout; importance vs load balancing.
- Combine tensor + dispatch tensor as einsums for a fully batched MoE forward.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def _sm(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def ref_route_top1(gate_logits, num_experts):
    a = np.asarray(gate_logits).argmax(axis=1)
    return a, np.bincount(a, minlength=num_experts)


def ref_aux_loss(gate_logits, num_experts):
    gate_logits = np.asarray(gate_logits, float)
    N = gate_logits.shape[0]
    p = _sm(gate_logits)
    f = np.bincount(p.argmax(1), minlength=num_experts) / N
    P = p.mean(0)
    return num_experts * float(np.sum(f * P))


def ref_aux_loss_backward(gate_logits, num_experts):
    gate_logits = np.asarray(gate_logits, float)
    N, E = gate_logits.shape
    p = _sm(gate_logits)
    f = np.bincount(p.argmax(1), minlength=E) / N      # treated as constant
    fp = p @ f                                          # (N,)  Σ_e f_e p[n,e]
    return (E / N) * p * (f[None, :] - fp[:, None])


def ref_moe_combine(gate_logits, expert_outs):
    p = _sm(np.asarray(gate_logits, float))
    return np.einsum("ne,end->nd", p, np.asarray(expert_outs, float))


def ref_moe_combine_backward(dy, gate_logits, expert_outs):
    dy = np.asarray(dy, float); expert_outs = np.asarray(expert_outs, float)
    p = _sm(np.asarray(gate_logits, float))
    d_expert = np.einsum("ne,nd->end", p, dy)
    dp = np.einsum("end,nd->ne", expert_outs, dy)
    dgate = p * (dp - (dp * p).sum(axis=1, keepdims=True))     # softmax backward over experts
    return dgate, d_expert


def ref_dispatch(gate_logits, capacity):
    gate_logits = np.asarray(gate_logits)
    N, E = gate_logits.shape
    a = gate_logits.argmax(1)
    onehot = (a[:, None] == np.arange(E)[None, :]).astype(int)
    cum = np.cumsum(onehot, axis=0)
    slot = cum[np.arange(N), a] - 1
    return slot < capacity, a, slot


_a, _c = ref_route_top1(np.eye(3), 3); assert list(_a) == [0, 1, 2]
assert np.isclose(ref_aux_loss(np.zeros((10, 2)), 2), 1.0)
_g = np.random.default_rng(9).standard_normal((15, 3))
assert np.allclose(ref_aux_loss_backward(_g, 3), _numgrad(lambda x: ref_aux_loss(x, 3), _g), atol=1e-5)
_eo = np.random.default_rng(8).standard_normal((3, 15, 4)); _dy = np.ones((15, 4))
_dg, _de = ref_moe_combine_backward(_dy, _g, _eo)
assert _dg.shape == (15, 3) and _de.shape == (3, 15, 4)
_k, _e, _s = ref_dispatch(np.array([[1., 0], [1., 0], [1., 0]]), 2)
assert list(_k) == [True, True, False]
print("reference solutions OK")